[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/01_Statistics_In_Learning.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/01_Statistics_In_Learning.ipynb)

# Statistics IS Learning — Not a Separate Step

Most courses teach statistics first, then machine learning as something different.
This notebook shows they are the **same thing**.

When a model trains, it computes statistics: means, variances, and
correlations between errors and features — and uses those numbers to improve.

---

**What this notebook covers**

1. The mean and variance of your target define your baseline difficulty
2. Correlation with the target tells you which features matter — before training
3. Gradient descent works by driving the correlation between errors and features to zero
4. When errors are uncorrelated with all features, learning is complete

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)
print('Ready.')

## Part 1 — The Target Is Your Problem Statement

Before touching features or models, look at the **target variable** — the thing you want to predict.

Its statistics tell you three things:
- what kind of prediction is needed (number vs category)
- how hard the problem is
- what a zero-effort baseline prediction looks like

In [ ]:
n = 300
study  = np.random.uniform(1, 10, n)
attend = np.random.uniform(50, 100, n)
prev   = np.random.uniform(40, 80, n)
noise  = np.random.normal(0, 4, n)

# Known true weights — learning should recover these
final_score = 10 + 2.5*study + 0.3*attend + 0.4*prev + noise

df = pd.DataFrame({
    'study_hours':    study,
    'attendance':     attend,
    'previous_score': prev,
    'final_score':    final_score
})

mean_score = df['final_score'].mean()
std_score  = df['final_score'].std()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['final_score'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(mean_score, color='orangered', lw=2.5, label=f'Mean = {mean_score:.1f}')
ax.set_xlabel('Final Score')
ax.set_title('Distribution of the target: final_score\nThe mean is your zero-effort starting prediction', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean:  {mean_score:.1f}  — predict this for everyone; wrong by ~{std_score:.1f} on average")
print(f"Std:   {std_score:.1f}  — this is the baseline error your model must beat")
print(f"Range: {df['final_score'].min():.1f} to {df['final_score'].max():.1f}")

## Part 2 — Correlation Tells You Which Features Matter

**Pearson correlation** measures how strongly each feature moves together with the target.
Range: −1 (perfect negative) to +1 (perfect positive). Near 0 = weak linear relationship.

> Compute correlations before training any model.
> A feature with near-zero correlation is unlikely to get a useful weight.
> A feature with high correlation will get a large weight — you can predict this before training.

In [ ]:
features = ['study_hours', 'attendance', 'previous_score']
corr_with_target = df[features + ['final_score']].corr()['final_score'].drop('final_score')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat in zip(axes, features):
    ax.scatter(df[feat], df['final_score'], alpha=0.35, s=16, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('final_score')
    r = corr_with_target[feat]
    ax.set_title(f'r = {r:.2f}', fontsize=13)

plt.suptitle('Feature vs Target — correlation is how linear the scatter cloud is', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("Correlations with final_score:")
for f, r in corr_with_target.items():
    print(f"  {f:<18} r = {r:.3f}")
print()
print("Higher |r| → stronger linear relationship → feature will get a meaningful weight")

## Part 3 — Gradient Descent Drives Error-Feature Correlation to Zero

Here is the core insight connecting statistics to learning:

> **The gradient for a weight is the correlation between the current errors and that feature (scaled by variance).**

When errors still correlate with a feature, the gradient is large — the weight must move.
When errors no longer correlate with a feature, the gradient is zero — that weight is done.

**Learning finishes when errors are uncorrelated with every feature.**
At that point the remaining error is pure noise — nothing more can be extracted.

In [ ]:
X_arr = df[features].values
y_arr = df['final_score'].values

w = np.zeros(3)
b = 0.0
lr = 0.0005
history = []

for step in range(300):
    pred  = X_arr @ w + b
    error = y_arr - pred

    corr_err = np.array([np.corrcoef(error, X_arr[:, i])[0, 1] for i in range(3)])
    history.append(corr_err.copy())

    grad_w = -(1/len(y_arr)) * (X_arr.T @ error)
    grad_b = -(1/len(y_arr)) * error.sum()
    w -= lr * grad_w
    b -= lr * grad_b

history = np.array(history)

fig, ax = plt.subplots(figsize=(10, 4))
for i, feat in enumerate(features):
    ax.plot(history[:, i], lw=2, label=feat)
ax.axhline(0, color='gray', lw=1.5, ls='--', label='zero — learning done')
ax.set_xlabel('Training step')
ax.set_ylabel('Correlation: error vs feature')
ax.set_title('Gradient descent eliminates the correlation between errors and features\nWhen all reach zero, learning is complete', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

print("Final learned weights:", {f: round(v, 3) for f, v in zip(features, w)})
print("True weights:          study_hours=2.5, attendance=0.3, previous_score=0.4")

## Key Takeaways

| Statistic | What it tells the model |
|---|---|
| Mean of target | Starting prediction (intercept begins here) |
| Std dev of target | How much error remains to reduce |
| Correlation (feature → target) | Which features will receive large weights |
| Correlation (error → feature) | The gradient — how much to adjust each weight right now |

**One sentence**: gradient descent removes the statistical link between errors and features, step by step, until only random noise remains.

The statistics you compute in EDA are not separate from learning — they preview exactly what the model will learn.